# Lab 27: The Search Ban

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 3, *The Attention Filter*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-27.ipynb)

**What you will do:** measure how your reaction time to a target at a previously cued location changes from faster to slower as more time passes, and watch the same explore-versus-exploit trade-off play out in a simple reinforcement-learning simulation.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 27 a friend hid a coin, and once you had searched a location you rarely returned to it
-- so much so that a coin secretly moved into an already-searched spot became hard to find
again. In the controlled online version, a flash draws attention to a location; if the target
then appears at that same location more than about 300 milliseconds later, responses are
*slower* than if it appears elsewhere. This is **inhibition of return (IOR)**: the brain briefly
tags a just-inspected location so that attention is biased towards new ground.

## Record your results

Find an online "inhibition of return" or "Posner cueing" task that lets you set the delay
(SOA) between the cue and the target. Run trials at several SOAs -- for example 100, 300,
600, 1000 and 1500 ms -- for targets that appear at the cued location and for targets that
appear elsewhere, and record your mean reaction time for each.

In [ ]:
soa_ms = [100, 300, 600, 1000, 1500]
rt_same_location = {100: 380, 300: 410, 600: 440, 1000: 460, 1500: 445}      # example data: replace with yours
rt_different_location = {100: 420, 300: 400, 600: 400, 1000: 405, 1500: 410}  # example data: replace with yours

results = pd.DataFrame({"same location (ms)": pd.Series(rt_same_location),
                         "different location (ms)": pd.Series(rt_different_location)})
results.index.name = "SOA (ms)"
display(results)
results.plot(marker="o", figsize=(6, 3.5))
plt.xlabel("time since the cue (ms)"); plt.ylabel("reaction time (ms)"); plt.show()

## Compare

The book says the effect should flip sign at around **300 ms**: faster responses to the cued
location before that (facilitation), slower responses after it (inhibition), with the
inhibitory tag lasting roughly **1-3 seconds** before it decays.

In [ ]:
results["IOR effect (ms)"] = results["same location (ms)"] - results["different location (ms)"]
display(results[["IOR effect (ms)"]])
crossing = results.index[(results["IOR effect (ms)"] > 0) & (results.index >= 300)]
print("Your IOR effect (positive = slower at the cued location) by SOA:")
print(results["IOR effect (ms)"].to_string())
if len(crossing):
    print(f"\nFirst positive (inhibited) value at or after 300 ms: {crossing.min()} ms SOA, "
          "consistent with the book's ~300 ms turning point.")
else:
    print("\nNo clear switch to inhibition in your data. Very approximate online timing, and only one")
    print("trial per condition, make this a noisy estimate -- more trials, or a purpose-built IOR task,")
    print("would sharpen it.")

## The AI side

The book maps inhibition of return onto the **exploration-exploitation trade-off** in
reinforcement learning: a system that only exploits what has already paid off will miss
better options, and a system that only explores will waste time on poor ones. Below, three
simple strategies play a "multi-armed bandit" -- repeatedly choosing among several unknown
options -- and only the ones with some form of built-in "inhibition of return" toward
recently-tried options reliably find the best one (Sutton & Barto, 2018, *Reinforcement
Learning: An Introduction*, 2nd ed., MIT Press).

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np

rng = np.random.default_rng(2026)
n_arms, n_steps = 8, 500
true_means = rng.uniform(0.2, 0.8, n_arms)  # unknown pay-off probabilities the agent must discover

def run_bandit(strategy, epsilon=0.1, c=2.0):
    counts = np.zeros(n_arms)
    values = np.zeros(n_arms)
    rewards = np.zeros(n_steps)
    for t in range(1, n_steps + 1):
        if strategy == "greedy":
            action = int(np.argmax(values))
        elif strategy == "epsilon_greedy":
            action = int(rng.integers(n_arms)) if rng.random() < epsilon else int(np.argmax(values))
        else:  # "ucb": prefers arms tried less often, like a decaying inhibition to revisit
            unexplored = np.where(counts == 0)[0]
            if len(unexplored):
                action = int(unexplored[0])
            else:
                bonus = c * np.sqrt(np.log(t) / counts)
                action = int(np.argmax(values + bonus))
        reward = float(rng.random() < true_means[action])
        counts[action] += 1
        values[action] += (reward - values[action]) / counts[action]
        rewards[t - 1] = reward
    return np.cumsum(rewards) / np.arange(1, n_steps + 1), counts

results_ai = {}
for strat in ["greedy", "epsilon_greedy", "ucb"]:
    avg_reward, counts = run_bandit(strat)
    results_ai[strat] = avg_reward
    found_best = int(np.argmax(counts)) == int(np.argmax(true_means))
    print(f"{strat:>15s}: settled on the best arm = {found_best}, "
          f"final average reward = {avg_reward[-1]:.2f} (best possible = {true_means.max():.2f})")

plt.figure(figsize=(7, 4))
for strat, avg_reward in results_ai.items():
    plt.plot(avg_reward, label=strat)
plt.axhline(true_means.max(), color="grey", linestyle="--", label="best possible")
plt.xlabel("step"); plt.ylabel("running average reward"); plt.legend()
plt.title("Exploration vs exploitation"); plt.show()

The "greedy" strategy never explores once it has an early favourite, so it can get stuck on a
mediocre option forever -- like a search with permanent, never-decaying inhibition of return
that can never re-check anywhere. Epsilon-greedy and UCB, like biological IOR, temporarily
avoid recently favoured options and usually find the best arm. The disanalogy: the brain's
inhibition is spatial and decays automatically after a few seconds regardless of outcome,
while these algorithms track value estimates and visit counts explicitly, with no notion of
physical location at all.

## Pool the class data

One person's IOR curve is noisy at a single trial per SOA. Pool the class: each person adds
rows with an anonymous ID, the SOA in ms, and their IOR effect (same-location RT minus
different-location RT, in ms), downloads it as CSV, and loads it here. The rows below are
placeholders so the cell runs.

In [ ]:
import io
csv_text = """id,soa_ms,ior_effect_ms
P01,100,-40
P01,300,10
P01,600,40
P01,1000,55
P01,1500,35
P02,100,-30
P02,300,-5
P02,600,25
P02,1000,45
P02,1500,30
P03,100,-50
P03,300,15
P03,600,35
P03,1000,50
P03,1500,20
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))
class_curve = df.groupby("soa_ms").ior_effect_ms.agg(["mean", "sem"])
display(class_curve)
plt.errorbar(class_curve.index, class_curve["mean"], yerr=class_curve["sem"], marker="o", color="firebrick")
plt.axhline(0, color="grey", linestyle="--")
plt.axvline(300, color="grey", linestyle=":", label="book's ~300 ms turning point")
plt.xlabel("SOA (ms)"); plt.ylabel("mean IOR effect (ms, same - different)"); plt.legend(); plt.show()

## Questions to think about

1. At what SOA did your own effect first turn positive (slower at the cued location)? How close was it to the book's approximately 300 ms estimate?
2. The book describes IOR as a form of "novelty bias" under predictive processing -- unchecked locations are more likely to contain new information. How does this framing explain why facilitation (faster responses) comes *before* inhibition, rather than the cue simply slowing everything down from the start?
3. In the AI Lab, the greedy strategy can get permanently stuck on a mediocre option. The book's Beyond the Lab section calls the human equivalent "premature closure" in medicine and "policy lock-in" in politics. What would the biological equivalent of "decaying inhibition" look like in one of those two domains -- what makes a closed decision reopen for reconsideration?
4. Patients with damage to the superior colliculus show reduced or absent IOR. Based on what the book says this structure does, why would losing it remove the tendency to avoid revisiting recently checked locations?

## Challenge

Test whether IOR affects accuracy as well as speed. Using the same online task, tally not just
reaction time but the number of missed or incorrect responses at the cued location versus
elsewhere, at an SOA past 300 ms.

In [ ]:
# Example data: replace with your own error counts at an SOA past 300 ms.
my_soa_ms = 600  # example data
my_errors = {"same location": 4, "different location": 1}  # example data, out of 20 trials each
my_trials_per_condition = 20

for condition, errors in my_errors.items():
    print(f"{condition}: {errors}/{my_trials_per_condition} errors "
          f"({errors / my_trials_per_condition:.0%}) at SOA = {my_soa_ms} ms")
print("If errors are also higher at the previously cued location, IOR is not just slowing you")
print("down -- it is making the previously checked location genuinely harder to process.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-27.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")